# A0 사전학습 Smoke Test

정식 A0 baseline 실험 전에 Colab 환경, 데이터, tokenizer, 모델, train loop, checkpoint 저장이 정상 동작하는지 확인합니다.

- 목적: 에러, NaN/Inf loss, CUDA OOM 여부 확인
- 기준 모델: `context_length=64`, `emb_dim=128`, `n_layers=2`, `n_heads=4`, `drop_rate=0.1`
- smoke 설정: 작은 데이터 일부만 사용, `batch_size=2`, `num_epochs=1`
- 주의: 이 결과는 정식 A0 결과표에 기록하지 않고, 실행 가능 여부 확인용으로만 사용합니다.

## 1. Colab 런타임 준비

먼저 Colab 메뉴에서 `Runtime > Change runtime type > GPU`를 선택하세요. 아래 셀은 저장소 루트로 이동합니다. 노트북을 GitHub/Colab에서 바로 열었다면 보통 자동으로 현재 폴더를 사용합니다.

In [3]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

def looks_like_repo(path):
    path = Path(path)
    return (path / "src").exists() and (path / "download_data.py").exists()

if looks_like_repo(Path.cwd()):
    repo_dir = Path.cwd()
elif IN_COLAB:
    repo_dir = Path("/content/gpt-lab")
    if not repo_dir.exists():
        repo_url = input("GitHub 저장소 URL을 입력하세요: ").strip()
        if repo_url.startswith("github.com/"):
            repo_url = "https://" + repo_url
        if not repo_url:
            raise ValueError("Colab에서는 저장소 URL이 필요합니다.")
        subprocess.run(["git", "clone", repo_url, str(repo_dir)], check=True)
else:
    repo_dir = Path.cwd()

os.chdir(repo_dir)
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

print("repo_dir:", repo_dir)
print("cwd:", Path.cwd())

repo_dir: /content/gpt-lab
cwd: /content/gpt-lab


## 2. 환경 확인과 패키지 설치

아래 출력은 실험 문서의 공통 환경 항목에 기록합니다.

In [5]:
!python --version
!nvidia-smi
!pip install -r requirements.txt

Python 3.12.13
/bin/bash: line 1: nvidia-smi: command not found


In [6]:
import math
import platform
import random
import time

import matplotlib
import numpy as np
import torch

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
except Exception:
    commit = "unknown"

print("python:", sys.version)
print("python_major_minor:", f"{sys.version_info.major}.{sys.version_info.minor}")
print("platform:", platform.platform())
print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
print("cuda_version:", torch.version.cuda)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("numpy:", np.__version__)
print("matplotlib:", matplotlib.__version__)
print("git commit:", commit)
print("device:", device)

python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
python_major_minor: 3.12
platform: Linux-6.6.122+-x86_64-with-glibc2.35
torch: 2.11.0+cpu
cuda_available: False
cuda_version: None
gpu: cpu
numpy: 2.0.2
matplotlib: 3.10.0
git commit: bab3567
device: cpu


## 3. 데이터 준비

NSMC 데이터를 내려받고 사전학습용 train/validation 텍스트 파일을 생성합니다.

In [5]:
import download_data

paths = download_data.main()

DATA_DIR = Path("data")
LM_TRAIN_PATH = DATA_DIR / "nsmc_lm_train.txt"
LM_VAL_PATH = DATA_DIR / "nsmc_lm_val.txt"

for path in [LM_TRAIN_PATH, LM_VAL_PATH]:
    print(path, path.exists(), path.stat().st_size if path.exists() else 0)

다운로드 중: https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt
저장됨: /content/gpt-lab/data/ratings_train.txt
다운로드 중: https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt
저장됨: /content/gpt-lab/data/ratings_test.txt
사전 학습 train 텍스트: /content/gpt-lab/data/nsmc_lm_train.txt (1,379,486자)
사전 학습 val 텍스트: /content/gpt-lab/data/nsmc_lm_val.txt (120,560자)
감성 분류 train: /content/gpt-lab/data/nsmc_sentiment_train.jsonl (137,996개)
감성 분류 val: /content/gpt-lab/data/nsmc_sentiment_val.jsonl (11,999개)
감성 분류 test: /content/gpt-lab/data/nsmc_sentiment_test.jsonl (49,997개)
data/nsmc_lm_train.txt True 3335336
data/nsmc_lm_val.txt True 291753


## 4. Baseline Config 고정

`BASE_CONFIG`와 `TRAIN_CONFIG`는 정식 실험에서도 공유하는 기준값입니다. `SMOKE_CONFIG`만 빠른 확인을 위해 작게 줄입니다.

In [6]:
BASE_CONFIG = {
    "vocab_size": 3000,
    "context_length": 64,
    "emb_dim": 128,
    "n_heads": 4,
    "n_layers": 2,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

TRAIN_CONFIG = {
    "seed": 42,
    "batch_size": 8,
    "learning_rate": 3e-4,
    "weight_decay": 0.0,
    "num_epochs": 2,
    "eval_freq": 100,
    "eval_iter": 10,
    "start_context": "영화",
}

SMOKE_CONFIG = {
    **TRAIN_CONFIG,
    "batch_size": 2,
    "num_epochs": 1,
    "eval_freq": 20,
    "eval_iter": 2,
    "train_char_limit": 25_000,
    "val_char_limit": 8_000,
}

print("BASE_CONFIG:", BASE_CONFIG)
print("TRAIN_CONFIG:", TRAIN_CONFIG)
print("SMOKE_CONFIG:", SMOKE_CONFIG)

BASE_CONFIG: {'vocab_size': 3000, 'context_length': 64, 'emb_dim': 128, 'n_heads': 4, 'n_layers': 2, 'drop_rate': 0.1, 'qkv_bias': False}
TRAIN_CONFIG: {'seed': 42, 'batch_size': 8, 'learning_rate': 0.0003, 'weight_decay': 0.0, 'num_epochs': 2, 'eval_freq': 100, 'eval_iter': 10, 'start_context': '영화'}
SMOKE_CONFIG: {'seed': 42, 'batch_size': 2, 'learning_rate': 0.0003, 'weight_decay': 0.0, 'num_epochs': 1, 'eval_freq': 20, 'eval_iter': 2, 'start_context': '영화', 'train_char_limit': 25000, 'val_char_limit': 8000}


## 5. Smoke Test 실행

작은 데이터 일부로 tokenizer, dataloader, 모델, optimizer, train loop, checkpoint 저장까지 확인합니다. 마지막에 `SMOKE TEST PASSED`가 나오면 정식 A0 baseline을 시작해도 됩니다.

In [7]:
from src.bpe import BPETokenizer
from src.dataset import create_dataloader
from src.model import GPTModel
from src.train import calc_loss_loader, train_model

def finite_or_raise(name, value):
    if not math.isfinite(value):
        raise RuntimeError(f"{name} is not finite: {value}")

started_at = time.time()
set_seed(SMOKE_CONFIG["seed"])

train_text = LM_TRAIN_PATH.read_text(encoding="utf-8")[:SMOKE_CONFIG["train_char_limit"]]
val_text = LM_VAL_PATH.read_text(encoding="utf-8")[:SMOKE_CONFIG["val_char_limit"]]

tokenizer_path = DATA_DIR / f"vocab_bpe_{BASE_CONFIG['vocab_size']}_smoke.json"
tokenizer = BPETokenizer(vocab_size=BASE_CONFIG["vocab_size"])
if tokenizer_path.exists():
    tokenizer.load(tokenizer_path)
    print("loaded tokenizer:", tokenizer_path)
else:
    tokenizer.train(train_text)
    tokenizer.save(tokenizer_path)
    print("saved tokenizer:", tokenizer_path)

train_ids = tokenizer.encode(train_text)
val_ids = tokenizer.encode(val_text)

train_loader = create_dataloader(
    train_ids,
    context_length=BASE_CONFIG["context_length"],
    batch_size=SMOKE_CONFIG["batch_size"],
    stride=BASE_CONFIG["context_length"],
    drop_last=True,
    shuffle=True,
)
val_loader = create_dataloader(
    val_ids,
    context_length=BASE_CONFIG["context_length"],
    batch_size=SMOKE_CONFIG["batch_size"],
    stride=BASE_CONFIG["context_length"],
    drop_last=False,
    shuffle=False,
)

print("train tokens:", len(train_ids), "train batches:", len(train_loader))
print("val tokens:", len(val_ids), "val batches:", len(val_loader))

model = GPTModel(BASE_CONFIG)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=SMOKE_CONFIG["learning_rate"],
    weight_decay=SMOKE_CONFIG["weight_decay"],
)

initial_val_loss = calc_loss_loader(
    val_loader,
    model,
    device,
    num_batches=SMOKE_CONFIG["eval_iter"],
)
finite_or_raise("initial_val_loss", initial_val_loss)
print(f"initial val loss: {initial_val_loss:.4f}")

history = {}
try:
    train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        device=device,
        num_epochs=SMOKE_CONFIG["num_epochs"],
        eval_freq=SMOKE_CONFIG["eval_freq"],
        eval_iter=SMOKE_CONFIG["eval_iter"],
        start_context=SMOKE_CONFIG["start_context"],
        tokenizer=tokenizer,
        ckpt_dir="checkpoints",
        experiment_id="A0_smoke",
        history=history,
    )
except torch.cuda.OutOfMemoryError as exc:
    raise RuntimeError("CUDA OOM during smoke test") from exc

for idx, loss in enumerate(history.get("train_losses", []), start=1):
    finite_or_raise(f"train_loss_epoch_{idx}", loss)
for idx, loss in enumerate(history.get("val_losses", []), start=1):
    finite_or_raise(f"val_loss_epoch_{idx}", loss)

if torch.cuda.is_available():
    print("cuda max memory allocated MB:", torch.cuda.max_memory_allocated() / 1024**2)

print("best val loss:", history.get("best_val_loss"))
print("best checkpoint:", history.get("best_checkpoint_path"))
print(f"elapsed sec: {time.time() - started_at:.1f}")
print("SMOKE TEST PASSED: no error, no NaN/Inf, no OOM")

saved tokenizer: data/vocab_bpe_3000_smoke.json
train tokens: 13671 train batches: 106
val tokens: 5079 val batches: 40
initial val loss: 8.2352
step 20: train loss 7.9352, val loss 8.1685
step 40: train loss 7.9032, val loss 8.0749
step 60: train loss 7.8592, val loss 7.9205
step 80: train loss 7.7198, val loss 7.6890
step 100: train loss 7.6061, val loss 7.4852
epoch 1: train loss 8.0007, val loss 7.4859, best val loss 7.4859
영화력를지....로하고도다� 몇도들 봤는데는데�를거,아면�는데어이가어가음요....나는는가재와하자고는에서도
고영화는데
어어니
best val loss: 7.485889029502869
best checkpoint: checkpoints/A0_smoke_20260602_best.pt
elapsed sec: 69.4
SMOKE TEST PASSED: no error, no NaN/Inf, no OOM


## 6. 통과 후 다음 단계

`SMOKE TEST PASSED`가 출력되면 정식 A0 baseline을 실행합니다. 정식 A0는 smoke 설정이 아니라 `TRAIN_CONFIG` 기준으로 `batch_size=8`, `num_epochs=2`, 전체 train/validation 데이터를 사용해 기록합니다.

## A0_basic 실험



In [10]:
import os
from getpass import getpass

os.environ["WANDB_API_KEY"] = getpass("W&B API key: ")

In [11]:
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: shawncan1573 (shawncan1573-discord) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [13]:
!cd /content/week14-team-05-gpt-lab
!git pull

/bin/bash: line 1: cd: /content/week14-team-05-gpt-lab: No such file or directory
Already up to date.


In [14]:
!python experiments/scripts/run_a_pretrain_stability.py \
  --experiment A0_basic \
  --train-char-limit 1500000 \
  --vocab-size 3000 \
  --wandb \
  --wandb-mode online

usage: run_a_pretrain_stability.py [-h]
                                   [--experiment {all,A0,A0_basic,A1,A2,A3,A4}]
                                   [--owner OWNER] [--date DATE]
                                   [--output-root OUTPUT_ROOT]
                                   [--vocab-size VOCAB_SIZE]
                                   [--tokenizer-path TOKENIZER_PATH]
                                   [--force-tokenizer-train] [--seed SEED]
                                   [--quick] [--install] [--skip-download]
                                   [--num-workers NUM_WORKERS]
                                   [--num-epochs NUM_EPOCHS]
                                   [--eval-iter EVAL_ITER] [--stride STRIDE]
                                   [--start-context START_CONTEXT]
                                   [--train-token-limit TRAIN_TOKEN_LIMIT]
                                   [--val-token-limit VAL_TOKEN_LIMIT]
                                   [--train-char-limit TRA